# Arrays in C

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Declare and fill fixed-size arrays, including 2D arrays, and pass them to functions together with their length.
- Explain why C never checks an index, and what an out-of-bounds access does.
- Choose where a large array lives: a local array on the stack, or a static or global one.

**In this notebook**, one idea at a time (run each cell before moving on):
1. Declaring an array
2. No .length: you keep the size
3. In memory: side by side
4. Passing an array to a function
5. C never checks the index
6. 2D arrays
7. Big arrays and the stack

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check_int and check_str print ✓ or ✗; the exercises below use them
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected %ld, got %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected \"%s\", got \"%s\"\n", what, expected, actual);
    }
}

## 1. Declaring an array

`int temps[7];` creates 7 ints side by side in memory. You can give the first values in braces: `int temps[7] = {21, 23, 19};` and the **rest start at 0**. With a full list you can leave the size out: `int primes[] = {2, 3, 5, 7};`. A local array with no initializer holds **garbage** until you fill it.

In [ ]:
int temps[7] = {21, 23, 19};
for (int i = 0; i < 7; i++) {
    printf("%d ", temps[i]);
}
printf("\n");
int primes[] = {2, 3, 5, 7};
primes[0] = 11;
printf("first: %d, last: %d\n", primes[0], primes[3]);

🧪 **Try it:** write `int zeros[5] = {0};` and print it: a common way to start an array at 0.

## 2. No .length: you keep the size

A C array **doesn't know its size**. Keep it yourself, usually in a constant: `#define DAYS 7` makes the compiler replace `DAYS` with `7` everywhere. Where the array is declared, `sizeof temps / sizeof temps[0]` also computes it: the total bytes divided by the bytes of one element.

In [ ]:
#include <stdio.h>
#define DAYS 7

int main(void) {
    int temps[DAYS] = {21, 23, 19, 25, 22, 20, 24};
    int n = sizeof temps / sizeof temps[0];
    printf("%zu bytes in total, %d elements\n", sizeof temps, n);
    int hot = 0;
    for (int i = 0; i < DAYS; i++) {
        if (temps[i] > 22) {
            hot++;
        }
    }
    printf("%d hot days\n", hot);
    return 0;
}

🧪 **Try it:** change `DAYS` to 5 and run: only one line changes, and the loop follows it.

## 3. In memory: side by side

The elements sit **next to each other** in memory, each one taking `sizeof` its type. That's how `a[i]` is found so fast: the address of `a[0]` plus `i` times the size of an element. The slides show it, and what happens one past the end.

#### 🎞️ An array in memory

![int a[4] holds four ints in a row at addresses 1000, 1004, 1008 and 1012; a[2] is at 1008, and a[4] reads past the end without any check](array-address-steps.svg)

In [ ]:
int a[4] = {10, 20, 30, 40};
for (int i = 0; i < 4; i++) {
    long offset = (char *) &a[i] - (char *) &a[0];
    printf("a[%d] = %d, %ld bytes after a[0]\n", i, a[i], offset);
}

🧪 **Try it:** make it a `double` array: how far apart are the elements now? And for `char`?

## 4. Passing an array to a function

A function gets the **array itself**, not a copy, so it can change the caller's elements. It doesn't get the size, so pass it as another parameter. `const` in the parameter promises the function only reads the array, and the compiler holds it to that promise.

In [ ]:
#include <stdio.h>

void double_all(int values[], int n) {
    for (int i = 0; i < n; i++) {
        values[i] *= 2;         // changes the caller's array
    }
}

void print_all(const int values[], int n) {
    for (int i = 0; i < n; i++) {
        printf("%d ", values[i]);
    }
    printf("\n");
}

int main(void) {
    int prices[] = {3, 5, 8};
    double_all(prices, 3);
    print_all(prices, 3);
    return 0;
}

🧪 **Try it:** try to change `values[0]` inside `print_all` and read the error: `const` forbids it.

### ✏️ Exercise: Sum an array

Return the sum of the first n values. A function can't ask an array its size, so n comes along.

In [ ]:
int sum_array(const int a[], int n) {
    // TODO: loop over the n values
    return 0;
}

int main(void) {
    int a[] = {3, 1, 4, 1, 5};
    check_int("five values", sum_array(a, 5), 14);
    check_int("only the first two", sum_array(a, 2), 4);
    check_int("none", sum_array(a, 0), 0);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
int sum_array(const int a[], int n) {
    int sum = 0;
    for (int i = 0; i < n; i++) {
        sum += a[i];
    }
    return sum;
}

int main(void) {
    int a[] = {3, 1, 4, 1, 5};
    check_int("five values", sum_array(a, 5), 14);
    check_int("only the first two", sum_array(a, 2), 4);
    check_int("none", sum_array(a, 0), 0);
    return 0;
}
```

</details>

### ✏️ Exercise: Count above a limit

Count the values greater than `limit` in the first n.

In [ ]:
int count_above(const int a[], int n, int limit) {
    // TODO: compare each value with limit
    return 0;
}

int main(void) {
    int a[] = {3, 9, 4, 12, 5};
    check_int("above 4", count_above(a, 5, 4), 3);
    check_int("above 20", count_above(a, 5, 20), 0);
    check_int("equal is not above", count_above(a, 1, 3), 0);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
int count_above(const int a[], int n, int limit) {
    int count = 0;
    for (int i = 0; i < n; i++) {
        if (a[i] > limit) {
            count++;
        }
    }
    return count;
}

int main(void) {
    int a[] = {3, 9, 4, 12, 5};
    check_int("above 4", count_above(a, 5, 4), 3);
    check_int("above 20", count_above(a, 5, 20), 0);
    check_int("equal is not above", count_above(a, 1, 3), 0);
    return 0;
}
```

</details>

## 5. C never checks the index

Java stops with an exception when an index is out of range. C **doesn't check**: `a[4]` in a 4-element array compiles and reads or writes whatever memory comes next. That's **undefined behavior**: the program may crash, corrupt other variables, or seem to work. Sphinx compiles with memory checks that catch it and point to the line, so you find it while learning.

In [ ]:
int a[4] = {10, 20, 30, 40};
int sum = 0;
for (int i = 0; i < 4; i++) {
    sum += a[i];
}
printf("sum = %d\n", sum);

🧪 **Try it:** change `i < 4` to `i <= 4` and run: read the memory check's report, which names the bad access and its line.

## 6. 2D arrays

`int grid[3][4];` is 3 rows of 4 ints, stored row after row. `grid[r][c]` reaches one cell, and two nested loops go through all of them. When you pass one to a function, the number of **columns** must be in the parameter: `void show(int g[][4], int rows)`.

In [ ]:
#include <stdio.h>

void show(int g[][4], int rows) {
    for (int r = 0; r < rows; r++) {
        for (int c = 0; c < 4; c++) {
            printf("%3d", g[r][c]);
        }
        printf("\n");
    }
}

int main(void) {
    int table[3][4];
    for (int r = 0; r < 3; r++) {
        for (int c = 0; c < 4; c++) {
            table[r][c] = (r + 1) * (c + 1);
        }
    }
    show(table, 3);
    return 0;
}

🧪 **Try it:** add the sum of each row at the end of its line.

## 7. Big arrays and the stack

A local array lives on the **stack**, which holds only a few megabytes. `int big[10000000];` inside a function can crash the program before it prints anything. For a big array, make it `static` (or global), which puts it in a separate area that lasts the whole program, or ask for heap memory with `malloc` (a later unit).

In [ ]:
static int counts[1000000];     // 4 MB, outside the stack, starts at 0
for (int i = 0; i < 1000000; i++) {
    counts[i % 10]++;
}
printf("counts[3] = %d\n", counts[3]);

🧪 **Try it:** note that a `static` array starts at 0 even without `= {0}`. Count how many of the numbers 0 to 999,999 end in 7 with it.

## In short

A C array is a fixed number of elements of one type, side by side in memory. It has no `.length`: you keep the size yourself, usually in a constant, and pass it to every function that takes the array.

```c
#include <stdio.h>

#define DAYS 7

double average(const int values[], int n) {
    int sum = 0;
    for (int i = 0; i < n; i++) {
        sum += values[i];
    }
    return (double) sum / n;
}

int main(void) {
    int temps[DAYS] = {21, 23, 19};      // the rest start at 0
    printf("%.2f\n", average(temps, DAYS));
    return 0;
}
```

A function gets the array itself, not a copy, so it can change its elements (`const` promises it won't). A local array lives on the **stack**, which holds only a few megabytes: make a very large array `static` or global.

**Watch out:** C never checks an index. `temps[7]` compiles, but it is undefined behavior: it may read or overwrite other data, crash, or seem to work. Sphinx's memory checks catch it and tell you the line.

## Readings

- [Arrays](https://beej.us/guide/bgc/html/split/arrays.html) (Beej's Guide to C Programming, chapter 6, 10 min): Getting the length with sizeof, initializers, "Out of Bounds!", multidimensional arrays, and passing arrays to functions. The pointer parts of 6.6 make full sense after the Pointers unit.
- [Lecture 2: Arrays](https://cs50.harvard.edu/x/2026/notes/2/#arrays) (CS50x (Harvard), lecture notes, 6 min): Optional: how an array sits in memory, one element after another, with pictures. Skip "Reading Levels" and "Debugging" at the top.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Histogram** · Easy
- **Sieve of Eratosthenes** · Easy
- **Ring Buffer** · Medium
- **Game of Life** · Medium
- **Big Factorial** · Hard